# Movie Recommendation System Pipeline

This notebook trains, evaluates, and demonstrates the `SequenceRecommender` model on MovieLens data. All text and comments are in English and code is standardized for readability.

In [4]:
# Config & Imports
import sys
import os
import torch
import torch.nn as nn
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('ggplot')

# Allow imports from repository root
if '.' not in sys.path:
    sys.path.append('.')

from dataset import load_data, process_data, collate_fn, MovieSequenceDataset
from model import SequenceRecommender
from train import train_with_oom_fallback
from evaluate import evaluate_model

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

# Notebook-configurable parameters
DATA_DIR = 'data'
CACHE_FILE = 'training_cache.pt'
FORCE_RETRAIN = False
START_BATCH_SIZE = 256
NUM_EPOCHS_QUICK = 5
MAX_SEQ_LEN = 50
ARCHITECTURES = ['rnn', 'lstm', 'gru']
OPTIMIZERS = ['sgd', 'adam']
LEARNING_RATES = [1e-2, 1e-3, 1e-4]


Using device: cuda


In [ ]:
# Interactive demo: Widget UI (English)
import ipywidgets as widgets
from IPython.display import display, clear_output

# Ensure `movies_df` is available. If not, load minimal data artifacts.
try:
    movies_df
except NameError:
    print('movies_df not found. Loading dataset (this may download MovieLens).')
    from dataset import get_dataloaders
    _, _, _, _, _, movie2idx, user2idx, movies_df = get_dataloaders(batch_size=START_BATCH_SIZE, max_seq_len=MAX_SEQ_LEN, data_dir=DATA_DIR)

# Build autocomplete options from titles
all_titles = movies_df['title'].dropna().unique().tolist()

# UI components
movie_input = widgets.Combobox(
    placeholder='Type a movie title (e.g. Toy Story)',
    options=all_titles,
    description='Movie:',
    ensure_option=False,
    disabled=False,
    layout=widgets.Layout(width='500px')
)

add_button = widgets.Button(description='Add to history', button_style='info', icon='plus')
predict_button = widgets.Button(description='Predict next movie', button_style='success', icon='magic')
clear_button = widgets.Button(description='Clear', button_style='warning', icon='refresh')

history_display = widgets.Output()
result_display = widgets.Output()

current_history = []

def add_movie(b):
    if movie_input.value:
        current_history.append(movie_input.value)
        with history_display:
            clear_output()
            print('Current history:', ', '.join(current_history))
        movie_input.value = ''

def clear_history(b):
    current_history.clear()
    movie_input.value = ''
    with history_display:
        clear_output()
        print('Current history: (empty)')
    with result_display:
        clear_output()

def run_prediction(b):
    with result_display:
        clear_output()
        if not current_history:
            print('Please add at least one movie to history before predicting.')
            return

        print(f'Predicting based on {len(current_history)} history items...')
        print('-' * 50)
        # Use model prediction if available, otherwise inform the user
        if 'predict_next_movie' in globals() and 'best_model' in globals():
            try:
                predict_next_movie(current_history, best_model)
            except Exception as e:
                print('Model prediction failed:', e)
                print('You can train/select a best_model first, or call the fallback predictor.')
        else:
            print('No trained model (`best_model`) available. Run the training/evaluation cells or use the fallback predictor.')

add_button.on_click(add_movie)
clear_button.on_click(clear_history)
predict_button.on_click(run_prediction)

with history_display:
    print('Current history: (empty)')

buttons_layout = widgets.HBox([add_button, clear_button, predict_button])
main_ui = widgets.VBox([
    widgets.HTML('<h2>🍿 Interactive Demo: Movie Recommendation</h2>'),
    widgets.HTML('<p><i>1. Type or pick a movie and click <b>Add to history</b>.<br>2. Click <b>Predict next movie</b> to run the model.</i></p>'),
    movie_input,
    buttons_layout,
    history_display,
    result_display
])

display(main_ui)


movies_df not found. Loading dataset (this may download MovieLens).
